In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras import layers, models
from PIL import ImageFile
import pandas as pd
import numpy as np
import shutil
import os

ImageFile.LOAD_TRUNCATED_IMAGES = True
datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

In [69]:
base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

modelVGG = models.Sequential()
modelVGG.add(base_model)
modelVGG.add(layers.GlobalMaxPooling2D())
modelVGG.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling2d_2          │ (None, 512)            │             0 │
│ (GlobalMaxPooling2D)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,714,688 (56.13 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 14,714,688 (56.13 MB)

In [3]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


# train + vali

In [4]:
trainValiPath='/content/gdrive/MyDrive/ml/train+vali'
trainValiImage = datagen.flow_from_directory(trainValiPath, target_size=(224,224), batch_size=32, class_mode='categorical', shuffle=False)
trainValifeatures = modelVGG.predict(trainValiImage, verbose=1)

Found 2954 images belonging to 18 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


93/93 ━━━━━━━━━━━━━━━━━━━━ 1478s 16s/step


In [5]:
trainValifilenames = trainValiImage.filenames
trainValifeatures = trainValifeatures / np.linalg.norm(trainValifeatures, axis=1, keepdims=True)
df_trainVali = pd.DataFrame(trainValifeatures)
df_trainVali.insert(0, "filename", trainValifilenames)
df_trainVali.to_csv("trainValiFeature.csv", index=False)

In [67]:
df_trainVali.to_csv("trainValiFeature.csv", index=False)
shutil.move('trainValiFeature.csv', '/content/gdrive/MyDrive/ml/vggFinetune/')

'/content/gdrive/MyDrive/ml/vggFinetune/trainValiFeature.csv'

# seperate train and vali

In [7]:
trainPath='/content/gdrive/MyDrive/train'
valiPath='/content/gdrive/MyDrive/val'

In [8]:
trainImage = datagen.flow_from_directory(trainPath, target_size=(224,224), batch_size=32, class_mode='categorical', shuffle=False )
valiImage= datagen.flow_from_directory(valiPath, target_size=(224,224), batch_size=32, class_mode='categorical', shuffle=False )

Found 2872 images belonging to 18 classes.
Found 761 images belonging to 18 classes.


In [9]:
trainLabel = []
valiLabel = []

for folder in sorted(os.listdir(trainPath)):
    folder_path = os.path.join(trainPath, folder)
    if os.path.isdir(folder_path):
        label = folder
        for file in sorted(os.listdir(folder_path)):
            if os.path.isfile(os.path.join(folder_path, file)):
                trainLabel.append((label, label+"/"+file))

for folder in sorted(os.listdir(valiPath)):
    folder_path = os.path.join(valiPath, folder)
    if os.path.isdir(folder_path):
        label = folder
        for file in sorted(os.listdir(folder_path)):
            if os.path.isfile(os.path.join(folder_path, file)):
                valiLabel.append((label, label+"/"+file))


In [10]:
train = pd.DataFrame(trainLabel, columns=['label', 'pathname'])
vali = pd.DataFrame(valiLabel, columns=['label', 'pathname'])

train.to_csv('trainLabel.csv', index=False)
vali.to_csv('valiLabel.csv', index=False)

In [11]:
train_features = modelVGG.predict(trainImage, verbose=1)
val_features = modelVGG.predict(valiImage, verbose=1)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


90/90 ━━━━━━━━━━━━━━━━━━━━ 1257s 14s/step
24/24 ━━━━━━━━━━━━━━━━━━━━ 328s 14s/step


In [12]:
train_filenames = trainImage.filenames
val_filenames = valiImage.filenames

train_features = train_features / np.linalg.norm(train_features, axis=1, keepdims=True)
val_features = val_features / np.linalg.norm(val_features, axis=1, keepdims=True)

df_train = pd.DataFrame(train_features)
df_val = pd.DataFrame(val_features)

df_train.insert(0, "filename", train_filenames)
df_val.insert(0, "filename", val_filenames)

df_train.to_csv("trainFeature.csv", index=False)
df_val.to_csv("valiFeature.csv", index=False)

In [71]:
import joblib

filename = 'vggFinetune.sav'
joblib.dump(modelVGG, filename)

['vggFinetune.sav']

In [77]:
shutil.move('vggFinetune.sav', '/content/gdrive/MyDrive/ml/vggFinetune/')

'/content/gdrive/MyDrive/ml/vggFinetune/vggFinetune.sav'

In [14]:
shutil.move('vggFinetune.sav', '/content/gdrive/MyDrive/ml/vggFinetune/')
shutil.move('trainFeature.csv', '/content/gdrive/MyDrive/ml/vggFinetune/')
shutil.move('trainLabel.csv', '/content/gdrive/MyDrive/ml/vggFinetune/')
shutil.move('valiLabel.csv', '/content/gdrive/MyDrive/ml/vggFinetune/')
shutil.move('valiFeature.csv', '/content/gdrive/MyDrive/ml/vggFinetune/')

'/content/gdrive/MyDrive/ml/vggFinetune/valiFeature.csv'

# End of VGG

In [17]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [18]:
trainFeature= pd.read_csv('/content/gdrive/MyDrive/ml/vggFinetune/trainFeature.csv')
valiFeature=pd.read_csv('/content/gdrive/MyDrive/ml/vggFinetune/valiFeature.csv')

trainFilename=trainFeature['filename']
valiFilename=valiFeature['filename']

trainFeature=trainFeature.drop(['filename'],axis=1)
valiFeature=valiFeature.drop(['filename'],axis=1)

In [19]:
from sklearn.preprocessing import StandardScaler
stdscaler = StandardScaler()
stdscaler.fit(trainFeature)

trainFeature = stdscaler.transform(trainFeature)
valiFeature = stdscaler.transform(valiFeature)

In [20]:
trainLabel= pd.read_csv('/content/gdrive/MyDrive/ml/vggFinetune/trainLabel.csv')
valiLabel = pd.read_csv('/content/gdrive/MyDrive/ml/vggFinetune/valiLabel.csv')

trainLabel=trainLabel.drop(['pathname'],axis=1)
valiLabel=valiLabel.drop(['pathname'],axis=1)

In [21]:
trainLabel=pd.get_dummies(trainLabel['label'])
valiLabel=pd.get_dummies(valiLabel['label'])
valiLabel=valiLabel.reindex(columns=trainLabel.columns, fill_value=0)

trainLabel = trainLabel.astype(int)
valiLabel = valiLabel.astype(int)

In [22]:
from tensorflow.keras.optimizers import Adam

In [23]:
model = models.Sequential()
model.add(layers.Dense(512, input_shape=(512,)))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))
model.add(layers.Dropout(0.3))

model.add(layers.Dense(512))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))

model.add(layers.Dense(18, activation='softmax'))
model.summary()

adamm = Adam(learning_rate=0.0001)
model.compile(optimizer=adamm, loss='categorical_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 18)             │         9,234 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 538,642 (2.05 MB)

 Trainable params: 536,594 (2.05 MB)

 Non-trainable params: 2,048 (8.00 KB)

In [24]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=20,
    restore_best_weights=True
)

In [25]:
history = model.fit(trainFeature, trainLabel, epochs=300,
                    validation_data=(valiFeature, valiLabel),
                    callbacks=[early_stop])

Epoch 1/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 6s 36ms/step - accuracy: 0.1904 - loss: 2.7957 - val_accuracy: 0.7162 - val_loss: 1.4474
Epoch 2/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7283 - loss: 1.2220 - val_accuracy: 0.8397 - val_loss: 0.7719
Epoch 3/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8362 - loss: 0.7744 - val_accuracy: 0.9067 - val_loss: 0.4978
Epoch 4/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8862 - loss: 0.5449 - val_accuracy: 0.9277 - val_loss: 0.3609
Epoch 5/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9117 - loss: 0.4200 - val_accuracy: 0.9501 - val_loss: 0.2734
Epoch 6/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9326 - loss: 0.3320 - val_accuracy: 0.9658 - val_loss: 0.2130
Epoch 7/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9512 - loss: 0.2747 - val_accuracy: 0.9790 - val_loss: 0.1726
Epoch 8/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9623 - loss: 0.2370 - val_accuracy: 0.9842 - 

In [26]:
new_model = models.Sequential(model.layers[:-1])

for old_layer, new_layer in zip(model.layers[:-1], new_model.layers):
    new_layer.set_weights(old_layer.get_weights())
new_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 512)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 529,408 (2.02 MB)

 Trainable params: 527,360 (2.01 MB)

 Non-trainable params: 2,048 (8.00 KB)

In [27]:
joblib.dump(new_model, 'cnnVGGFinetune.sav')

['cnnVGGFinetune.sav']

# Train done

In [2]:
import joblib
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [10]:
stdscaler = joblib.load('/content/gdrive/MyDrive/ml/vggFinetune/stdScaler.pkl')
new_model = joblib.load('/content/gdrive/MyDrive/ml/vggFinetune/cnnVGGFinetune.sav')

In [5]:
trainValiFeature= pd.read_csv('/content/gdrive/MyDrive/ml/vggFinetune/trainValiFeature.csv')
trainValiFilename=trainValiFeature['filename']
trainValiFeature=trainValiFeature.drop(['filename'],axis=1)

In [6]:
trainValiFeature

,0,1,2,3,4,5,6,7,8,9,...,502,503,504,505,506,507,508,509,510,511
0,-1.000000,-1.000000,-0.157252,-0.765096,-0.488506,-0.323950,-0.206493,-1.000000,-0.948474,-1.000000,...,-0.392888,-0.358822,-0.723405,-0.953562,-0.108846,-1.000000,-0.577604,-1.000000,-0.868214,-0.016333
1,-0.522175,-1.000000,-0.039439,-0.066807,-1.000000,-0.807060,-0.623140,-1.000000,-0.745502,-1.000000,...,-0.562845,-1.000000,-1.000000,-1.000000,-0.167435,-1.000000,-1.000000,-1.000000,-0.588960,-0.842353
2,-1.000000,-1.000000,-0.235506,-0.216966,-0.320379,-0.258085,-0.339403,-1.000000,-1.000000,-0.947223,...,-1.000000,-0.103487,-1.000000,-1.000000,-0.239508,-1.000000,-1.000000,-1.000000,0.162415,-0.916725
3,-0.431705,-1.000000,-0.769801,0.182114,-0.502354,-0.222787,0.411158,-1.000000,-1.000000,-1.000000,...,-1.000000,-1.000000,-1.000000,-0.473655,-1.000000,-0.975148,-1.000000,-1.000000,-0.339571,0.431471
4,-1.000000,-1.000000,0.293264,-0.776204,-0.578216,-0.064811,0.535574,-1.000000,-1.000000,-1.000000,...,-1.000000,0.019303,-0.704580,-1.000000,-0.800924,-1.000000,-0.208503,-1.000000,-0.302543,0.230251
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2949,-0.880191,-0.236778,-1.000000,-0.999384,-1.000000,-0.055308,-0.761383,-0.671464,-1.000000,-0.515047,...,-1.000000,-0.787704,-0.673174,-1.000000,-0.096399,-0.570922,-0.656192,-1.000000,-1.000000,-1.000000
2950,-0.729827,-1.000000,-1.000000,-0.037956,-1.000000,-0.798766,-1.000000,-0.890580,-1.000000,-0.572972,...,-1.000000,-1.000000,-1.000000,-1.000000,-0.914348,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
2951,-0.254159,-1.000000,-1.000000,-0.656521,-1.000000,-0.491337,-0.897433,-0.966847,-1.000000,-0.536661,...,-1.000000,-1.000000,0.433464,-0.410266,0.339561,-1.000000,-1.000000,-1.000000,-0.387281,-1.000000
2952,-0.354492,-0.706416,-1.000000,-0.953001,-0.682930,-0.912922,-1.000000,-1.000000,-1.000000,-0.135260,...,-1.000000,-1.000000,-1.000000,-1.000000,-0.851866,-1.000000,-1.000000,-0.979488,-1.000000,-1.000000


In [11]:
trainValiFeatures = stdscaler.transform(trainValiFeature)
trainVali = new_model.predict(trainValiFeatures, verbose=1)

93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


In [12]:
trainVali.shape

(2954, 512)

In [13]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(trainVali)

trainVali = scaler.transform(trainVali)

In [14]:
trainVali

array([[-0.34737003, -0.6445805 , -1.        , ..., -1.        ,
        -1.        , -1.        ],
       [-0.43039036, -0.5432135 , -1.        , ..., -1.        ,
        -1.        , -0.9688397 ],
       [-0.26680005,  0.01919281, -1.        , ..., -1.        ,
        -1.        , -1.        ],
       ...,
       [-0.62608695, -0.32899624, -1.        , ...,  0.16461694,
        -1.        , -1.        ],
       [-0.01622617, -0.52669036, -1.        , ...,  0.07064176,
        -1.        , -1.        ],
       [ 0.02203572, -0.73713374, -1.        , ..., -0.5421676 ,
        -1.        , -1.        ]], dtype=float32)

In [15]:
df_trainVali = pd.DataFrame(trainVali)
df_trainVali.insert(0, "filename", trainValiFilename)

In [16]:
joblib.dump(scaler, 'minMaxScaler.pkl')

['minMaxScaler.pkl']

In [17]:
df_trainVali.to_csv("trainValiVectors.csv", index=False)

In [18]:
import joblib
df_trainVali.to_csv("trainValiVectors.csv", index=False)
joblib.dump(scaler, 'minMaxScaler.pkl')
joblib.dump(new_model, 'cnnVGGFinetune.sav')
joblib.dump(stdscaler, 'stdScaler.pkl')

['stdScaler.pkl']

In [21]:
shutil.move('cnnVGGFinetune.sav', '/content/gdrive/MyDrive/ml/vggFinetune/')
shutil.move('minMaxScaler.pkl', '/content/gdrive/MyDrive/ml/vggFinetune/')
shutil.move('stdScaler.pkl', '/content/gdrive/MyDrive/ml/vggFinetune/')
shutil.move('trainValiVectors.csv', '/content/gdrive/MyDrive/ml/vggFinetune/')

'/content/gdrive/MyDrive/ml/vggFinetune/trainValiVectors.csv'